WHAT'S HAPPENING?
----------------
Neural networks use decimal numbers (Float32). Each number takes 4 bytes.
Quantization converts them to integers (Int8 or Int4). Each number takes 1 byte or 0.5 bytes.
Same information, 4x to 8x less space. That's it.


THE MAPPING (HOW DECIMALS BECOME INTEGERS)?
You can't just chop off decimals. You need a conversion rule.

Float range: -1.0 to 1.0 (your model weights)
Int8 range: -128 to 127 (what integers can store)

Scale = (Float_Max - Float_Min) / (Int_Max - Int_Min)
Scale = 2.0 / 255 = 0.00784

To quantize: Int = round(Float / Scale)
To dequantize: Float = Int * Scale

Example:
Float: 0.5
Int: round(0.5 / 0.00784) = 64
Back to Float: 64 * 0.00784 = 0.502 (close enough)


MEMORY REPRESENTATION (RAW BITS)
--------------------------------
This is what the computer actually stores on the RAM stick.

Float32 (Value: 0.5)
Hex: 0x3F000000
Binary: 00111111 00000000 00000000 00000000
Breakdown: Sign (0) | Exponent (01111110) | Mantissa (00000000000000000000000)
Size: 4 bytes per number

Int8 (Value: 64)
Hex: 0x40
Binary: 01000000
Breakdown: 8-bit signed integer
Size: 1 byte per number

Int4 (Value: 7 and 12 packed)
Hex: 0x7C
Binary: 0111 1100
Breakdown: First 4 bits (0111 = 7), Second 4 bits (1100 = 12)
Size: 0.5 bytes per number (2 values packed into 1 byte)

When you quantize, you change the bit pattern entirely. 
The CPU reads different instructions for Float32 vs Int8. 
Int8 math is simpler, so the CPU can do more of it per second.

HOW IT'S STORED?
You don't just store the integers. You store:
- The quantized values (int8 or int4)
- The scale factor (float32, one per group of weights)
- The zero point (int8, one per group of weights)

When you need the original values:
1. Load the int values
2. Multiply by scale
3. Add zero point
4. You get back approximately the original float

You only dequantize when you actually need the values.
Otherwise, computation happens directly on int8/int4 (faster).

AWQ (ACTIVATION-AWARE WEIGHT QUANTIZATION)
------------------------------------------
AWQ is quantization designed specifically for Large Language Models (LLMs).
Standard quantization fails on LLMs because of outliers.

The Outlier Problem:
In LLMs, most weights are small (near 0). 
But a few weights are huge (outliers). 
These huge weights are critical. They control important features.
If you force them into Int4, you lose precision on the most important numbers.
The model becomes dumb.

AWQ Solution:
1. Analyze Activations: Run data through the model. See which weights produce large activation values.
2. Identify Salient Channels: Mark the top 1% of weight channels as "salient" (important).
3. Protect Salient Weights: Do not quantize these aggressively. Keep them at higher precision or scale them differently.
4. Quantize the Rest: Compress the remaining 99% of weights to Int4.
5. Group Scaling: Instead of one scale for the whole layer, use one scale per group of 128 weights. This adapts to local changes in data distribution.

Why Group Size 128?
Weights aren't uniform across the whole layer. 
Some groups of 128 weights might range from -0.1 to 0.1. 
Others might range from -0.5 to 0.5.
One global scale would be inaccurate. 
Per-group scaling fits the data tighter.

Memory Layout of AWQ:
- Weights: Packed Int4 (4 bits per weight)
- Scales: Float16 (one per 128 weights)
- Zero Points: Int16 (one per 128 weights)

The overhead of scales is tiny compared to the savings on weights.

PRODUCTION CODE (AUTOAWQ)
-------------------------
This is how you actually quantize a model like Mistral or Llama in production.
Library: autoawq (standard for AWQ)

from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

# 1. Load the pre-trained model (Float16 usually)
model_path = "mistralai/Mistral-7B-v0.1"
model = AutoAWQForCausalLM.from_pretrained(model_path, device_map="auto", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# 2. Configure Quantization
quant_config = {
    "zero_point": True,      # Use zero point scaling
    "q_group_size": 128,     # Scale per 128 weights
    "w_bit": 4,              # 4-bit weights
    "version": "GEMM"        # Kernel type for inference
}

# 3. Run Quantization
# This scans activations and applies AWQ protection
print("Quantizing model...")
model.quantize(tokenizer, quant_config=quant_config)

# 4. Save the Quantized Model
model.save_quantized("./mistral-7b-awq-4bit")
tokenizer.save_pretrained("./mistral-7b-awq-4bit")

# 5. Load and Inference Later
# Loads directly into VRAM as 4-bit, no dequantization overhead during load
model = AutoAWQForCausalLM.from_quantized("./mistral-7b-awq-4bit", device_map="auto")

prompt = "Explain quantization in simple terms."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(output[0]))


CALIBRATION DATA
AWQ needs a small dataset to measure activations.
Usually 128 to 512 samples from Wikipedia or C4 dataset.

Example calibration loop (simplified):
for batch in calibration_data:
    inputs = tokenizer(batch, return_tensors="pt").to(model.device)
    model(inputs)  # Forward pass only, no gradients
    # Hooks record activation magnitudes during this pass

This takes 5-10 minutes on a GPU.
Once done, the quantization parameters are fixed.
You don't need calibration data for inference, only for the initial quantization.
